# Why `CHLA_ooi_profiles_plus_PACE.parquet` is 42% duplicate rows

A step-by-step walkthrough of the duplication in the published **Ocean Observatories
Initiative** (OOI) half of the CHLA-Z training set, traced back to its root cause in
`fish-pace-datasets/datasets/chla_z/notebooks/ooi.ipynb`.

Outline:

1. Load the parquet and count distinct `profile_id`s
2. Look at a duplicate group — the OOI side of each row is byte-identical
3. What *does* differ: the **PACE** match-up columns (granule file, time window)
4. The `nrt` flag does not explain it — most duplicates are two *standard* granules
5. Why one OOI row matches two granules: midnight timestamps sit in the overlap of consecutive DAY granules
6. The lines in `ooi.ipynb` / `ml_utils.py` responsible
7. The notebook already contains the fix — it computes `df_dedup` and then saves `df_merged`
8. Summary and the one-line fix

Everything below is computed live from the parquet on disk; nothing is hard-coded.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

HERE = Path(".")            # contributor_folders/charlie/
REPO = HERE / ".." / ".."   # 2026-go-bgc-hackweek repo root

# The published training parquet, copied here so this notebook runs on a bare clone.
# Source: fish-pace/chla-z, brt_training_data/CHLA_ooi_profiles_plus_PACE.parquet
PARQUET = HERE / "data/CHLA_ooi_profiles_plus_PACE.parquet"

# The two creation notebooks, copied into upstream/ with outputs stripped (source only).
# Source: fish-pace/fish-pace-datasets @ 9e65251, datasets/chla_z/notebooks/
OOI_NB = HERE / "upstream/ooi.ipynb"
ARGO_NB = HERE / "upstream/argo-matchups.ipynb"

# ml_utils.py at this repo's root is byte-identical to chla-z/notebooks/ml_utils.py.
ML_UTILS = REPO / "ml_utils.py"

for p in (PARQUET, OOI_NB, ARGO_NB, ML_UTILS):
    assert p.exists(), p


## 1. Load the parquet and count distinct `profile_id`s

`profile_id` is `<dataset_id>_<YYYYMMDD>` — one OOI instrument-day. The parquet is
described as one row per profile, so `profile_id` should be unique.

In [2]:
df = pd.read_parquet(PARQUET)
for col in ["time", "pace_Rrs_t_start", "pace_Rrs_t_end"]:
    df[col] = pd.to_datetime(df[col], utc=True)

chla_cols = [c for c in df.columns if c.startswith("CHLA_")]
rrs_cols = [c for c in df.columns if c.startswith("pace_Rrs_") and c[-1].isdigit()]
pace_meta = [c for c in df.columns if c.startswith("pace_") and c not in rrs_cols]

n_rows, n_ids = len(df), df["profile_id"].nunique()
print(f"rows            : {n_rows:,}")
print(f"distinct ids    : {n_ids:,}")
print(f"surplus rows    : {n_rows - n_ids:,}  ({(n_rows - n_ids) / n_rows:.1%} of the file)")
print(f"CHLA bin columns: {len(chla_cols)}   pace Rrs columns: {len(rrs_cols)}   pace meta: {pace_meta}")

rows            : 10,989
distinct ids    : 6,415
surplus rows    : 4,574  (41.6% of the file)
CHLA bin columns: 20   pace Rrs columns: 172   pace meta: ['pace_Rrs_file', 'pace_Rrs_t_start', 'pace_Rrs_t_end', 'pace_Rrs_lat', 'pace_Rrs_lon', 'pace_chlor_a', 'pace_Kd_490']


In [3]:
group_size = df.groupby("profile_id").size()
dup_ids = group_size[group_size > 1].index

summary = group_size.value_counts().sort_index().rename("n_profile_ids").to_frame()
summary["rows"] = summary.index * summary["n_profile_ids"]
summary.index.name = "rows per profile_id"
print(f"{len(dup_ids):,} of {n_ids:,} ids appear more than once")
summary

3,444 of 6,415 ids appear more than once


,n_profile_ids,rows
rows per profile_id,,
1,2971,2971
2,2869,5738
3,36,108
4,523,2092
5,16,80


## 2. A duplicate group — the OOI side is byte-identical

Pick one `profile_id` that appears exactly twice and look at the OOI columns.

In [4]:
two_ids = group_size[group_size == 2].index
example_id = two_ids[0]
ooi_cols = ["profile_id", "time", "lat", "lon"] + chla_cols[:6] + ["description"]
df.loc[df["profile_id"] == example_id, ooi_cols]

,profile_id,time,lat,lon,CHLA_0_10,CHLA_10_20,CHLA_20_30,CHLA_30_40,CHLA_40_50,CHLA_50_60,description
484,ooi-ce01issp-sp001-08-flortj000_20240418,2024-04-18 00:00:00+00:00,44.65845,-124.0979,0.330928,0.311356,0.27878,NaN,NaN,NaN,Coastal Endurance: Oregon Inshore Surface Pier...
505,ooi-ce01issp-sp001-08-flortj000_20240418,2024-04-18 00:00:00+00:00,44.65845,-124.0979,0.330928,0.311356,0.27878,NaN,NaN,NaN,Coastal Endurance: Oregon Inshore Surface Pier...


In [5]:
# Is that true for *every* duplicate group? Count distinct values per OOI column within each id.
dups = df[df["profile_id"].isin(dup_ids)]
ooi_side = ["time", "lat", "lon"] + chla_cols
distinct_per_group = dups.groupby("profile_id")[ooi_side].nunique(dropna=False)
n_groups_varying = (distinct_per_group.max(axis=1) > 1).sum()
print(f"duplicate groups whose OOI columns (time, lat, lon, 20 CHLA bins) vary at all: {n_groups_varying} / {len(dup_ids):,}")

duplicate groups whose OOI columns (time, lat, lon, 20 CHLA bins) vary at all: 0 / 3,444


So on the OOI side the key **is** unique: every duplicate group carries one instrument-day
measured once. The multiplication must come from the other half of the row.

## 3. What differs: the PACE match-up columns

In [6]:
show = ["profile_id", "time", "pace_Rrs_file", "pace_Rrs_t_start", "pace_Rrs_t_end", "nrt", "pace_chlor_a"]
df.loc[df["profile_id"] == example_id, show]

,profile_id,time,pace_Rrs_file,pace_Rrs_t_start,pace_Rrs_t_end,nrt,pace_chlor_a
484,ooi-ce01issp-sp001-08-flortj000_20240418,2024-04-18 00:00:00+00:00,PACE_OCI.20240417.L3m.DAY.RRS.V3_1.Rrs.4km.nc,2024-04-17 00:26:59.092000+00:00,2024-04-18 01:16:59.878000+00:00,False,1.577286
505,ooi-ce01issp-sp001-08-flortj000_20240418,2024-04-18 00:00:00+00:00,PACE_OCI.20240418.L3m.DAY.RRS.V3_1.Rrs.4km.nc,2024-04-17 23:33:40.084000+00:00,2024-04-19 01:36:59.965000+00:00,False,0.991646


In [7]:
# Across all duplicate groups: which columns take >1 value within a group?
varying = (dups.groupby("profile_id")[pace_meta + ["nrt"]]
               .nunique(dropna=False)
               .gt(1)
               .mean()
               .rename("fraction of dup groups where this column varies"))
varying.to_frame()

,fraction of dup groups where this column varies
pace_Rrs_file,1.000000
pace_Rrs_t_start,0.887340
pace_Rrs_t_end,0.891986
pace_Rrs_lat,0.000000
pace_Rrs_lon,0.000000
pace_chlor_a,0.414925
pace_Kd_490,0.413763
nrt,0.269164


Each duplicate row is the same OOI instrument-day joined to a **different PACE granule**
(`pace_Rrs_file`, and hence `pace_Rrs_t_start` / `t_end`). The join is one-to-many where
it was meant to be one-to-one.

## 4. The `nrt` flag does not explain it

A plausible story: each day has a standard (refined) granule *and* a **near-real-time**
(NRT) granule, and both were matched. That accounts for some groups — but not most.

In [8]:
nrt_pattern = (dups.groupby("profile_id")["nrt"]
                   .apply(lambda s: tuple(sorted(s)))
                   .value_counts()
                   .rename("n_profile_ids"))
nrt_pattern.to_frame()

,n_profile_ids
nrt,
"(False, False)",2252
"(False, False, True, True)",523
"(False, True)",388
"(True, True)",229
"(False, False, False)",36
"(False, False, True, True, True)",16


In [9]:
std = df[~df["nrt"]]
ids_in_multiple_std = (std.groupby("profile_id").size() > 1).sum()
print(f"profile_ids matched to MORE THAN ONE *standard* (non-NRT) granule: {ids_in_multiple_std:,} of {n_ids:,}")

profile_ids matched to MORE THAN ONE *standard* (non-NRT) granule: 2,827 of 6,415


The dominant pattern is `(False, False)` — the **same OOI day matched to two different
standard granules**. NRT/standard overlap is the minority case (the `(False, True)` pairs
and the `True` members of the 4- and 5-row groups).

## 5. Why one OOI day matches two standard granules

Look at the two granules in the example group: their names are consecutive days, and
their time-coverage windows **overlap around 00:00 UTC**.

In [10]:
ex = df.loc[df["profile_id"] == example_id, ["time", "pace_Rrs_file", "pace_Rrs_t_start", "pace_Rrs_t_end"]].sort_values("pace_Rrs_t_start")
ex = ex.assign(t_start_plus_24h=ex["pace_Rrs_t_start"] + pd.Timedelta(hours=24))
ex

,time,pace_Rrs_file,pace_Rrs_t_start,pace_Rrs_t_end,t_start_plus_24h
484,2024-04-18 00:00:00+00:00,PACE_OCI.20240417.L3m.DAY.RRS.V3_1.Rrs.4km.nc,2024-04-17 00:26:59.092000+00:00,2024-04-18 01:16:59.878000+00:00,2024-04-18 00:26:59.092000+00:00
505,2024-04-18 00:00:00+00:00,PACE_OCI.20240418.L3m.DAY.RRS.V3_1.Rrs.4km.nc,2024-04-17 23:33:40.084000+00:00,2024-04-19 01:36:59.965000+00:00,2024-04-18 23:33:40.084000+00:00


In [11]:
# Which window rule did the match-up use? Test both candidates against every row in the file.
in_t_end_window = ((df["time"] >= df["pace_Rrs_t_start"]) & (df["time"] < df["pace_Rrs_t_end"])).mean()
in_24h_window = ((df["time"] >= df["pace_Rrs_t_start"]) & (df["time"] < df["pace_Rrs_t_start"] + pd.Timedelta(hours=24))).mean()
print(f"rows with t_start <= time < t_end           : {in_t_end_window:.4%}")
print(f"rows with t_start <= time < t_start + 24 h  : {in_24h_window:.4%}")

rows with t_start <= time < t_end           : 97.9434%
rows with t_start <= time < t_start + 24 h  : 100.0000%


In [12]:
# Every published OOI row is stamped at exactly midnight UTC (the notebook's daily mean).
tod = df["time"].dt.hour * 60 + df["time"].dt.minute
print("distinct time-of-day values (minutes after 00:00 UTC):", sorted(tod.unique()))

distinct time-of-day values (minutes after 00:00 UTC): [np.int32(0)]


In [13]:
# Which granule *days* pair up inside a multi-standard-granule group?  Offset = granule's
# filename day minus the OOI day.  Expect {-1, 0}: "yesterday's" granule and "today's".
std = std.assign(
    granule_day=pd.to_datetime(std["pace_Rrs_file"].str.extract(r"PACE_OCI\.(\d{8})")[0], utc=True),
)
std = std.assign(offset_days=(std["granule_day"] - std["time"].dt.normalize()).dt.days)
multi = std[std["profile_id"].isin((std.groupby("profile_id").size() > 1).pipe(lambda s: s[s].index))]
(multi.groupby("profile_id")["offset_days"]
      .apply(lambda s: tuple(sorted(s)))
      .value_counts()
      .rename("n_profile_ids")
      .to_frame())

,n_profile_ids
offset_days,
"(-1, 0)",2791
"(-2, -1, 0)",36


For a midnight-stamped OOI day *D* to fall into both windows:

* granule *D−1* must start **after** its own 00:00 UTC (then `t_start + 24 h` reaches past midnight of *D*), and
* granule *D* must start **before** its own 00:00 UTC (i.e. late on *D−1*).

PACE DAY granules start either side of midnight — the `time_coverage_start` of a daily L3
composite is the first swath that contributes, which wanders by ±30 min around 00:00 UTC.

In [14]:
granules = std.drop_duplicates("pace_Rrs_file")
start_offset_min = ((granules["pace_Rrs_t_start"] - granules["granule_day"]).dt.total_seconds() / 60).round()
print(f"{len(granules)} standard granules; share whose t_start is BEFORE their own 00:00 UTC: {(start_offset_min < 0).mean():.0%}")
(start_offset_min.value_counts(bins=[-60, -30, 0, 30, 60, 120, 1600]).sort_index()
                 .rename("n_granules").rename_axis("t_start minus granule-day midnight (min)").to_frame())

560 standard granules; share whose t_start is BEFORE their own 00:00 UTC: 36%


,n_granules
t_start minus granule-day midnight (min),
"(-60.001, -30.0]",6
"(-30.0, 0.0]",200
"(0.0, 30.0]",229
"(30.0, 60.0]",109
"(60.0, 120.0]",2
"(120.0, 1600.0]",14


In [15]:
# Footnote: the 36 three-standard-granule groups come from one upstream oddity — a granule whose
# time_coverage_start lies a day *after* its filename day with a ~16-minute window, so its 24 h
# match window reaches two midnights ahead. Same bug, one extra granule.
triple_ids = multi.groupby("profile_id").size().pipe(lambda s: s[s == 3].index)
std.loc[std["profile_id"] == triple_ids[0], ["profile_id", "time", "pace_Rrs_file", "pace_Rrs_t_start", "pace_Rrs_t_end"]]

,profile_id,time,pace_Rrs_file,pace_Rrs_t_start,pace_Rrs_t_end
242,ooi-ce02shsp-sp001-07-flortj000_20240401,2024-04-01 00:00:00+00:00,PACE_OCI.20240330.L3m.DAY.RRS.V3_1.Rrs.4km.nc,2024-03-31 00:25:16.624000+00:00,2024-03-31 00:41:24.004000+00:00
252,ooi-ce02shsp-sp001-07-flortj000_20240401,2024-04-01 00:00:00+00:00,PACE_OCI.20240331.L3m.DAY.RRS.V3_1.Rrs.4km.nc,2024-03-31 00:25:16.624000+00:00,2024-04-01 02:24:44.883000+00:00
262,ooi-ce02shsp-sp001-07-flortj000_20240401,2024-04-01 00:00:00+00:00,PACE_OCI.20240401.L3m.DAY.RRS.V3_1.Rrs.4km.nc,2024-03-31 23:33:05.173000+00:00,2024-04-02 01:36:28.915000+00:00


Put together:

* the OOI notebook stamps every row at **00:00 UTC** (`dt.floor("D")` daily mean);
* the match-up assigns a row to every granule whose window `[t_start, t_start + 24 h)` contains it;
* PACE DAY granules start a little before or after 00:00 UTC, so when granule *D−1* starts after midnight and granule *D* starts before it, the two 24 h windows both contain 00:00 UTC of day *D*;
* ⇒ a midnight-stamped row usually lands in two consecutive standard granules.

Argo profiles, with times spread through the day, rarely hit the overlap — which is why the
same match-up code did not visibly misbehave for the Argo half.

## 6. The responsible lines in `ooi.ipynb` and `ml_utils.py`

Read straight from the files on disk so the quotes stay honest.

In [16]:
import json
import re


def nb_code_cells(path: Path) -> list[str]:
    nb = json.loads(Path(path).read_text())
    return ["".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code"]


def show_lines(src: str, patterns: list[str], label: str, context: int = 0) -> None:
    lines = src.splitlines()
    hits = [i for i, line in enumerate(lines) if any(re.search(p, line) for p in patterns)]
    print(f"--- {label} ---")
    shown: set[int] = set()
    for i in hits:
        for j in range(max(0, i - context), min(len(lines), i + context + 1)):
            if j not in shown:
                print(f"{j + 1:4d}: {lines[j]}")
                shown.add(j)
    print()


ooi_cells = nb_code_cells(OOI_NB)
argo_cells = nb_code_cells(ARGO_NB)


def find_cell(cells: list[str], must_contain: str) -> str:
    matches = [c for c in cells if must_contain in c]
    assert matches, must_contain
    return matches[0]

In [17]:
# (a) Daily-mean aggregation → every row is stamped at midnight
save_fn = find_cell(ooi_cells, "def ooi_mooring_save_file")
show_lines(save_fn, [r'dt\.floor\("D"\)', r'groupby\(\["date", "depth_bin"\]', r'rename\(columns=\{"date": "time"\}'],
           "ooi.ipynb — ooi_mooring_save_file()")

--- ooi.ipynb — ooi_mooring_save_file() ---
  24:     df["date"] = df["time"].dt.floor("D")
  47:         .groupby(["date", "depth_bin"], observed=True)
  70:     wide = wide.rename(columns={"date": "time"})



In [18]:
# (b) The match-up window in ml_utils.one_file_matches: [t_start, t_start + 24 h)
src = ML_UTILS.read_text()
start = src.index("def one_file_matches(")
body = src[start:start + 6000]
show_lines(body, [r"t_start = pd\.to_datetime", r"t_end_24 =", r"df_record = df\["], "ml_utils.py — one_file_matches()")

--- ml_utils.py — one_file_matches() ---
 108:         t_start = pd.to_datetime(ds.attrs["time_coverage_start"], utc=True)
 119:         t_end_24 = t_start + pd.Timedelta(hours=24)
 120:         df_record = df[(df_times >= t_start) & (df_times < t_end_24)]



In [19]:
# (c) Standard + NRT granule lists are concatenated, and every granule is matched independently
search_cell = find_cell(ooi_cells, "PACE_OCI_L3M_RRS_NRT")
show_lines(search_cell, [r"short_name", r"results = rrs_results \+ rrs_results_nrt"], "ooi.ipynb — granule search")
run_batch_cell = find_cell(ooi_cells, "def run_batch")
show_lines(run_batch_cell, [r"for i, f in enumerate\(fileset\)", r"one_file_matches\(", r"df_plus\.append"], "ooi.ipynb — run_batch()")

--- ooi.ipynb — granule search ---
  10:     short_name = "PACE_OCI_L3M_RRS",
  15:     short_name = "PACE_OCI_L3M_RRS_NRT",
  19: results = rrs_results + rrs_results_nrt

--- ooi.ipynb — run_batch() ---
  43:     for i, f in enumerate(fileset):
  44:         df_record, pts = mu.one_file_matches(
  61:         df_plus.append(df_record_plus)



## 7. The notebook already computes the fix — and then saves the wrong frame

The "Cleaned" cell builds `df_dedup`: per `profile_id`, keep the non-NRT row, then the row
whose granule window centre is nearest the profile time. The **next** cell writes
`df_merged` — the raw one-to-many table — to `data/CHLA_ooi_profiles_plus_PACE.parquet`.

In [20]:
clean_cell = find_cell(ooi_cells, "Duplicates replaced with the PACE data that is closest to TIME")
print(clean_cell)

# Cleaned; 
# Duplicates replaced with the PACE data that is closest to TIME

import pandas as pd
from pathlib import Path
results_dir = Path("_temp_data/matchups/ooi")
platform = "ooi"
var = "CHLA"

df_merged = pd.read_parquet(f"{results_dir}/{var}_{platform}_Rrs_chlor_a_Kd_all.parquet")

df = df_merged

# Parse all times as UTC-aware datetimes
for col in ["time", "pace_Rrs_t_start", "pace_Rrs_t_end"]:
    df[col] = pd.to_datetime(df[col], utc=True)

# is file NRT
df["nrt"] = [("NRT" in f) for f in df["pace_Rrs_file"]]
idx = df.groupby("profile_id")["nrt"].idxmin()
df = df.loc[idx].drop(columns=["nrt"]).reset_index(drop=True)

# Just in case there are 2 files due to slightly overlapping time windows
# though turns out there were not any
# center of the PACE time window
df["pace_center"] = df["pace_Rrs_t_start"] + (
    df["pace_Rrs_t_end"] - df["pace_Rrs_t_start"]
) / 2

# 2. absolute time difference between profile TIME and window center
df["time_diff"] = (df["time"] - df["pace_cente

In [21]:
save_cell = find_cell(ooi_cells, 'out_path = "data/CHLA_ooi_profiles_plus_PACE.parquet"')
print(save_cell)

out_path = "data/CHLA_ooi_profiles_plus_PACE.parquet"
df_merged.to_parquet(out_path)


Two details worth pointing at:

* `df = df_merged` is an **alias**, so `df["nrt"] = …` adds the `nrt` column to `df_merged`
  itself — which is why the published file carries `nrt` but none of the de-duplication.
  The subsequent `df = df.loc[idx]…` rebinding does not touch `df_merged`.
* The comment "Just in case there are 2 files due to slightly overlapping time windows —
  though turns out there were not any" was true for **Argo** and is false for OOI (section 5).

Compare the Argo notebook, from which `ooi.ipynb` was adapted: it has one extra cell that
re-merges `df_dedup` back into `df_merged` before saving, so there `df_merged` *is* de-duplicated.

In [22]:
argo_merge_cell = find_cell(argo_cells, "profiles.merge(df_dedup")
print("argo-matchups.ipynb — the cell ooi.ipynb lacks:\n")
print(argo_merge_cell)

argo-matchups.ipynb — the cell ooi.ipynb lacks:

# Merge
keys = ["profile_id", "PLATFORM_NUMBER", "CYCLE_NUMBER",
        "TIME", "LATITUDE", "LONGITUDE"]
df_merged = profiles.merge(df_dedup, on=keys, how="inner")
# fix col order
cols = df_merged.columns.tolist()
cols = ["profile_id"] + [c for c in cols if c != "profile_id"]
df_merged = df_merged[cols].sort_values(by="TIME").reset_index(drop=True)
df_merged.shape


### Proof: apply the notebook's own `df_dedup` logic to the published parquet

In [23]:
d = df.copy()
idx = d.groupby("profile_id")["nrt"].idxmin()           # prefer the non-NRT granule
d = d.loc[idx].reset_index(drop=True)
d["pace_center"] = d["pace_Rrs_t_start"] + (d["pace_Rrs_t_end"] - d["pace_Rrs_t_start"]) / 2
d["time_diff"] = (d["time"] - d["pace_center"]).abs()
idx = d.groupby("profile_id")["time_diff"].idxmin()     # then the nearest window centre
df_dedup = d.loc[idx].drop(columns=["pace_center", "time_diff"]).sort_values("time").reset_index(drop=True)

print(f"published rows      : {len(df):,}")
print(f"after cell-43 dedup : {len(df_dedup):,}")
print(f"distinct profile_id : {df_dedup['profile_id'].nunique():,}")
print(f"any duplicate ids?  : {df_dedup['profile_id'].duplicated().any()}")

published rows      : 10,989
after cell-43 dedup : 6,415
distinct profile_id : 6,415
any duplicate ids?  : False


## 8. Summary

| Finding | Value |
|---|---|
| Published rows | 10,989 |
| Distinct `profile_id` | 6,415 |
| Duplicate groups (2–5 rows) | 3,444 |
| OOI-side columns vary within a group | never |
| Groups that are two *standard* granules `(False, False)` | 2,252 (+ 523 `(F,F,T,T)` + 16 `(F,F,T,T,T)` + 36 `(F,F,F)`) |
| Ids matched to >1 standard granule | 2,827 |
| Rows after the notebook's own `df_dedup` | 6,415, no duplicates |

**Root cause (two defects that compound):**

1. `ooi.ipynb` computes `df_dedup` and then saves `df_merged` — the one-line bug is
   `df_merged.to_parquet(out_path)` → `df_dedup.to_parquet(out_path)`. (The Argo notebook
   avoids it only because it has an extra merge cell.)
2. There is something to de-duplicate because every OOI row is stamped at 00:00 UTC by the
   daily-mean step, which is exactly where the 24 h match windows `[t_start, t_start + 24 h)`
   of consecutive PACE DAY granules overlap (2,827 ids sit in both "yesterday's" and
   "today's" standard granule); the standard+NRT concatenation adds a second, smaller
   source of duplicates.

**Consequences for anyone training on the published file:** identical rows fall on both
sides of a random split (leakage on ~42% of the OOI half) and double-weight their site-days;
row-count-based balance figures overstate unique OOI data by ~1.7×. Minimal mitigation
without a rebuild: `drop_duplicates("profile_id")` (or the `df_dedup` logic above) before
splitting.

The per-event rebuild in this experiment (`README.md`, stage D) removes the midnight
stamp — which moves most events out of the overlap band — and must still resolve the
remaining overlap-band events the way the `df_dedup` cell intended (nearest window centre).